In [37]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier # import BaggingClassifier for classification problems, and BaggingRegressor for regression problems
from sklearn.svm import SVC, SVR


import warnings
warnings.filterwarnings('ignore')

In [38]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=10000, n_features=10, n_informative=3, random_state=42)
print(X.shape, y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred)}")

ans1 = accuracy_score(y_test, y_pred)

(10000, 10) (10000,)
Decision Tree Accuracy: 0.9265


### **Bagging**

In [39]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,       # The number of individual base models to train (creating 500 different Decision Trees)
    max_samples=0.5,        # The number of samples to draw from X to train each base estimator. Instead of giving all the dataset, we are giving each tree only 50% of the data (randomly selected). This ensures that the trees are different from one another (diversity)
    bootstrap=True,         # Enables "Sampling with Replacement."
    random_state=42
)

# so here we are assigning 500 decision tree classifier models, where each model handles 50% of the dataset randomly 

bag.fit(X_train, y_train)
y_pred = bag.predict(X_test)
print(f"Bagging Accuracy: {accuracy_score(y_test, y_pred)}")

ans2 = accuracy_score(y_test, y_pred)

Bagging Accuracy: 0.95


- 500 Separate Models: The code creates 500 independent DecisionTreeClassifier instances.
- Random 50% Slices: Every one of those 500 trees is given a different random subset of your data (containing 50% of the original rows).
- The "Bootstrap" Twist: Because bootstrap=True (which is the default), when a tree picks its 50%, it can pick the same row more than once (sampling with replacement).

In [40]:
print(bag.estimators_samples_[0].shape)
print(bag.estimators_features_[0].shape)

(4000,)
(10,)


### **Bagging using SVM**

In [41]:
bag2 = BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)
     
bag2.fit(X_train,y_train)
y_pred = bag2.predict(X_test)
print("Bagging using SVM",accuracy_score(y_test,y_pred))

ans3 = accuracy_score(y_test, y_pred)

Bagging using SVM 0.9125


### **Pasting**

In [42]:
bag3 = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,
    random_state=42,
    # verbose = 1,        # Controls the verbosity when fitting and predicting.
    n_jobs= -1           # Training 500 decision trees can take a long time if done one by one. n_jobs tells Scikit-Learn how many CPU cores to use.
)


bag3.fit(X_train,y_train)
y_pred = bag3.predict(X_test)
print("Pasting classifier",accuracy_score(y_test,y_pred))

ans4 = accuracy_score(y_test,y_pred)


Pasting classifier 0.946


1. **n_jobs (Parallel Processing)**

Training 500 decision trees can take a long time if done one by one. n_jobs tells Scikit-Learn how many CPU cores to use.

- n_jobs = 1 (Default): The computer uses only one CPU core. It trains Tree #1, then Tree #2, then Tree #3, and so on.
- n_jobs = 2: Use two cores. It trains two trees at the exact same time.
- n_jobs = -1: This is the most common setting for big models. It tells the computer to use ALL available cores in your processor.

2. **verbose (Progress Logging)**

By default, Scikit-Learn is "silent"—you run a cell, and it just shows a busy symbol until it's finished. verbose changes this by giving you updates.

- verbose = 0 (Default): No output is shown during training.
- verbose = 1: Shows a progress bar or simple text updates (e.g., "Building 500 estimators...").
- verbose > 1: The higher the number, the more detailed the reports will be.

### **Random Subspaces**

In [43]:
bag4 = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)

bag4.fit(X_train,y_train)
y_pred = bag4.predict(X_test)
print("Random Subspaces classifier",accuracy_score(y_test,y_pred))

ans5 = accuracy_score(y_test,y_pred)

Random Subspaces classifier 0.9415


### **Random Patches**

In [44]:
bag5 = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)

bag5.fit(X_train,y_train)
y_pred = bag5.predict(X_test)
print("Random Patches classifier",accuracy_score(y_test,y_pred))

ans6 = accuracy_score(y_test,y_pred)

Random Patches classifier 0.938


### **OOB Score**

In [45]:
bag6 = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

bag6.fit(X_train,y_train)
y_pred = bag6.predict(X_test)
print("Accuracy",accuracy_score(y_test,y_pred))

ans7 = accuracy_score(y_test,y_pred)


Accuracy 0.945


Keep in mind before running all the above models at once, coz we are doing ensembing. So it will take a hell lot of time to compute all the models. 

Do it accordingly.

In [47]:
# printing all the models accuracy

print(f"Decision Tree \t\t: {ans1}")
print(f"Bagging \t\t: {ans2}")
print(f"SVM \t\t\t: {ans3}")
print(f"Pasting \t\t: {ans4}")
print(f"Random Subspaces \t: {ans5}")
print(f"Random Patches \t\t: {ans6}")
print(f"OOB Score \t\t: {ans7}")

Decision Tree 		: 0.9265
Bagging 		: 0.95
SVM 			: 0.9125
Pasting 		: 0.946
Random Subspaces 	: 0.9415
Random Patches 		: 0.938
OOB Score 		: 0.945
